# KKBox Churn Prediction - EDA

            This notebook is a project-showcase overview of the data used to train the churn model.

            **Goal:** understand dataset health, churn prevalence, temporal snapshot safety, and the strongest behavioral patterns before modeling.

            **Important:** raw KKBox files are large, especially `user_logs.csv`; this notebook reads compact Parquet artifacts generated by the pipeline whenever possible.

## Setup

            The notebook assumes the pipeline has already produced `data/interim/modeling_frame.parquet` and `data/processed/feature_frame.parquet`.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
MODELS = PROJECT_ROOT / "models"

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

## Artifact Inventory

In [ ]:
def inventory(paths):
    rows = []
    for base in paths:
        for path in sorted(base.glob("*")):
            if path.is_file():
                rows.append({
                    "directory": str(path.parent.relative_to(PROJECT_ROOT)),
                    "file": path.name,
                    "size_mb": path.stat().st_size / 1024**2,
                    "modified": pd.Timestamp(path.stat().st_mtime, unit="s"),
                })
    return pd.DataFrame(rows).sort_values(["directory", "file"])

inventory([DATA_RAW, DATA_INTERIM, DATA_PROCESSED, REPORTS, MODELS])

## Load Modeling Artifacts

In [ ]:
modeling_path = DATA_INTERIM / "modeling_frame.parquet"
feature_path = DATA_PROCESSED / "feature_frame.parquet"

modeling_frame = pd.read_parquet(modeling_path)
feature_frame = pd.read_parquet(feature_path)

print("modeling_frame:", modeling_frame.shape)
print("feature_frame :", feature_frame.shape)
display(modeling_frame.head())

## Target Balance

In [ ]:
target_summary = (
    feature_frame["is_churn"]
    .value_counts(dropna=False)
    .rename_axis("is_churn")
    .reset_index(name="rows")
)
target_summary["rate"] = target_summary["rows"] / target_summary["rows"].sum()
display(target_summary)

ax = sns.barplot(data=target_summary, x="is_churn", y="rate", hue="is_churn", legend=False)
ax.set_title("Churn class balance")
ax.set_ylabel("Share of users")
ax.set_xlabel("is_churn")
plt.show()

## Data Quality Snapshot

In [ ]:
missing = (
    feature_frame.isna()
    .mean()
    .sort_values(ascending=False)
    .head(20)
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "feature"})
)
display(missing)

ax = sns.barplot(data=missing, y="feature", x="missing_rate", color="#4c78a8")
ax.set_title("Top missing-value rates")
ax.set_xlabel("Missing rate")
ax.set_ylabel("")
plt.show()

## Key Behavioral Segments

In [ ]:
segment_cols = [
    "auto_renew_rate",
    "cancel_rate",
    "days_since_last_transaction",
    "days_since_last_log",
    "active_days_30d",
    "total_secs_30d",
    "completion_rate_30d",
    "recency_weighted_total_secs",
]
available = [col for col in segment_cols if col in feature_frame.columns]
display(feature_frame.groupby("is_churn")[available].median().T.rename(columns={0: "not_churn", 1: "churn"}))

## Auto-Renew Is Predictive, But It Is Not The Whole Story

In [ ]:
if "auto_renew_rate" in feature_frame.columns:
    buckets = pd.cut(
        feature_frame["auto_renew_rate"],
        bins=[-0.001, 0, 0.5, 0.999, 1.0],
        labels=["0", "(0, 0.5]", "(0.5, <1)", "1"],
    )
    auto_renew_summary = (
        feature_frame.assign(auto_renew_bucket=buckets)
        .groupby("auto_renew_bucket", observed=False)
        .agg(rows=("is_churn", "size"), churn_rate=("is_churn", "mean"))
        .reset_index()
    )
    display(auto_renew_summary)

    ax = sns.barplot(data=auto_renew_summary, x="auto_renew_bucket", y="churn_rate", color="#f58518")
    ax.set_title("Churn rate by auto-renew bucket")
    ax.set_xlabel("auto_renew_rate bucket")
    ax.set_ylabel("Churn rate")
    plt.show()

## Recency And Listening Momentum

In [ ]:
plot_cols = [
    "days_since_last_log",
    "days_since_last_transaction",
    "secs_7d_vs_prior_23d_ratio",
    "recency_weighted_total_secs",
]
plot_cols = [col for col in plot_cols if col in feature_frame.columns]

for col in plot_cols:
    sample = feature_frame[[col, "is_churn"]].dropna().sample(
        n=min(50_000, feature_frame[col].notna().sum()),
        random_state=42,
    )
    ax = sns.histplot(data=sample, x=col, hue="is_churn", bins=50, stat="density", common_norm=False)
    ax.set_title(f"Distribution by churn: {col}")
    plt.show()

## Correlation View For Selected Numeric Signals

In [ ]:
candidate_corr_cols = [
    "is_churn",
    "auto_renew_rate",
    "cancel_rate",
    "trans_count",
    "total_spend",
    "days_since_last_transaction",
    "days_since_last_log",
    "active_days_30d",
    "total_secs_30d",
    "secs_7d_vs_prior_23d_ratio",
    "recency_weighted_total_secs",
    "recency_weighted_usage_to_spend",
]
corr_cols = [col for col in candidate_corr_cols if col in feature_frame.columns]
corr = feature_frame[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f", square=False)
plt.title("Selected feature correlations")
plt.show()

## EDA Takeaways

            - The target is imbalanced, so AUC-PR and lift are better monitoring metrics than accuracy.
            - Renewal behavior is highly predictive, which is expected for subscription churn.
            - Listening recency, activity momentum, and value-perception features provide non-renewal alternatives for the model.
            - All downstream modeling should use the fixed cutoff date in `config/config.yaml` to avoid temporal leakage.